In [1]:
import pandas as pd
import numpy as np

# Loading the cleaned data
orders = pd.read_csv("data/raw/orders.csv")
events = pd.read_csv("data/raw/events.csv")
sessions = pd.read_csv("data/raw/sessions.csv")
products = pd.read_csv("data/raw/products.csv")

print("Data loaded successfully.")
print(f"Orders: {len(orders):,}")
print(f"Events: {len(events):,}")
print(f"Sessions: {len(sessions):,}")
print(f"Products: {len(products):,}")

Data loaded successfully.
Orders: 1,705
Events: 135,897
Sessions: 76,709
Products: 36


In [2]:
# Revenue Impact Simulation
# This simulation quantifies additional annual revenue from reducing
# Electronics cart abandonment by 5%, 10%, and 15%

# Step 1 - Establish current Electronics baseline metrics
electronics_products = products[
    products['category'] == 'Electronics']['product_id'].tolist()

# All cart events for Electronics
electronics_cart_events = events[
    (events['event_type'] == 'add_to_cart') &
    (events['product_id'].isin(electronics_products))
].copy()

# Sessions that completed a purchase
purchase_session_ids = set(
    events[events['event_type'] == 'purchase']['session_id'])

# Electronics cart sessions that did NOT purchase
electronics_abandoned = electronics_cart_events[
    ~electronics_cart_events['session_id'].isin(
        purchase_session_ids)]

# Electronics cart sessions that DID purchase
electronics_converted = electronics_cart_events[
    electronics_cart_events['session_id'].isin(
        purchase_session_ids)]

# Baseline metrics
total_electronics_carts = len(electronics_cart_events)
total_abandoned = len(electronics_abandoned)
total_converted = len(electronics_converted)
current_abandonment_rate = total_abandoned / total_electronics_carts
current_conversion_rate = total_converted / total_electronics_carts
avg_abandoned_price = electronics_abandoned['price_at_event'].mean()
total_lost_revenue = electronics_abandoned['price_at_event'].sum()
avg_order_value = orders[
    orders['product_id'].isin(electronics_products)][
    'order_value'].mean()

print("=" * 55)
print("NORTHGATE MARKETPLACE — ELECTRONICS FUNNEL BASELINE")
print("=" * 55)
print(f"Total Electronics cart sessions:    {total_electronics_carts:>6,}")
print(f"Converted to purchase:              {total_converted:>6,}")
print(f"Abandoned cart:                     {total_abandoned:>6,}")
print(f"Current abandonment rate:           {current_abandonment_rate:>6.1%}")
print(f"Current cart conversion rate:       {current_conversion_rate:>6.1%}")
print(f"Average abandoned cart value:       {avg_abandoned_price:>7.2f}")
print(f"Total annual lost revenue:          ${total_lost_revenue:>10,.2f}")
print(f"Average Electronics order value:    ${avg_order_value:>10,.2f}")

NORTHGATE MARKETPLACE — ELECTRONICS FUNNEL BASELINE
Total Electronics cart sessions:       754
Converted to purchase:                 163
Abandoned cart:                        591
Current abandonment rate:            78.4%
Current cart conversion rate:        21.6%
Average abandoned cart value:        102.82
Total annual lost revenue:          $ 60,764.54
Average Electronics order value:    $    133.85


In [4]:
# Revenue Recovery Simulation
# This simulation models additional annual revenue from reducing
# Electronics cart abandonment by 5%, 10%, and 15%

print("=" * 55)
print("NORTHGATE MARKETPLACE — REVENUE RECOVERY SIMULATION")
print("=" * 55)
print(f"Baseline: {total_abandoned} abandoned Electronics carts")
print(f"Average abandoned cart value: ${avg_abandoned_price:.2f}")
print(f"Total annual lost revenue: ${total_lost_revenue:,.2f}")
print()

recovery_scenarios = [0.05, 0.10, 0.15]
scenario_labels = ['Conservative (5%)', 
                   'Moderate (10%)', 
                   'Aggressive (15%)']

print(f"{'Scenario':<25} {'Carts Recovered':>15} "
      f"{'Additional Revenue':>20} "
      f"{'New Annual Revenue':>20}")
print("-" * 82)

for rate, label in zip(recovery_scenarios, scenario_labels):
    carts_recovered = int(total_abandoned * rate)
    additional_revenue = carts_recovered * avg_abandoned_price
    new_total_revenue = orders['order_value'].sum() + additional_revenue
    
    add_rev_str = f"${additional_revenue:,.2f}"
    new_rev_str = f"${new_total_revenue:,.2f}"
    
    print(f"{label:<25} {carts_recovered:>15,} "
          f"{add_rev_str:>20} "
          f"{new_rev_str:>20}")

print()
print("Note: Projections assume recovered carts convert at")
print("the average abandoned cart value of "
      f"${avg_abandoned_price:.2f}.")
print("Actual recovery value may vary based on intervention")
print("type and customer segment targeted.")

NORTHGATE MARKETPLACE — REVENUE RECOVERY SIMULATION
Baseline: 591 abandoned Electronics carts
Average abandoned cart value: $102.82
Total annual lost revenue: $60,764.54

Scenario                  Carts Recovered   Additional Revenue   New Annual Revenue
----------------------------------------------------------------------------------
Conservative (5%)                      29            $2,981.68          $122,536.81
Moderate (10%)                         59            $6,066.17          $125,621.30
Aggressive (15%)                       88            $9,047.85          $128,602.98

Note: Projections assume recovered carts convert at
the average abandoned cart value of $102.82.
Actual recovery value may vary based on intervention
type and customer segment targeted.
